# EDA: SBA Lending

Purpose: inspect national SBA 7(a) lending activity, lender concentration, county/sector coverage, and geography quality before blending with QCEW growth.

## Setup

Install if needed:

```bash
pip install pandas numpy matplotlib snowflake-connector-python
```

In [ ]:

from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

DATABASE = "SMB_MARKET_INTELLIGENCE_DEV"
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE", "COMPUTE_WH")
ROLE = os.getenv("SNOWFLAKE_ROLE", "ACCOUNTADMIN")

def get_connection():
    """Create a Snowflake connection from environment variables.

    Required environment variables for password auth:
        SNOWFLAKE_ACCOUNT
        SNOWFLAKE_USER
        SNOWFLAKE_PASSWORD

    Optional:
        SNOWFLAKE_AUTHENTICATOR=externalbrowser
        SNOWFLAKE_WAREHOUSE=COMPUTE_WH
        SNOWFLAKE_ROLE=ACCOUNTADMIN
    """
    import snowflake.connector

    account = os.getenv("SNOWFLAKE_ACCOUNT")
    user = os.getenv("SNOWFLAKE_USER")
    password = os.getenv("SNOWFLAKE_PASSWORD")
    authenticator = os.getenv("SNOWFLAKE_AUTHENTICATOR")

    if not account or not user:
        raise RuntimeError(
            "Set SNOWFLAKE_ACCOUNT and SNOWFLAKE_USER before running this notebook. "
            "For browser auth, also set SNOWFLAKE_AUTHENTICATOR=externalbrowser. "
            "For password auth, set SNOWFLAKE_PASSWORD."
        )

    kwargs = {
        "account": account,
        "user": user,
        "warehouse": WAREHOUSE,
        "database": DATABASE,
        "role": ROLE,
    }

    if authenticator:
        kwargs["authenticator"] = authenticator
    elif password:
        kwargs["password"] = password
    else:
        raise RuntimeError(
            "Set either SNOWFLAKE_PASSWORD or SNOWFLAKE_AUTHENTICATOR=externalbrowser."
        )

    return snowflake.connector.connect(**kwargs)

def read_sql(query):
    with get_connection() as conn:
        return pd.read_sql(query, conn)

def show_basic_profile(df):
    display(df.head())
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")


## Load SBA lending aggregate

In [ ]:

sba = read_sql("""
SELECT
    state_fips,
    state_abbr,
    state_name,
    county_fips,
    qcew_industry_code,
    sector_name,
    year,
    quarter,
    period_id,
    sba_loan_count,
    sba_gross_approval_amount,
    sba_initial_approval_amount,
    sba_current_approval_amount,
    sba_jobs_supported,
    active_lender_count,
    loans_missing_lender_name,
    avg_gross_loan_amount,
    top_lender_name,
    top_lender_loan_count,
    top_lender_gross_approval_amount,
    top_lender_share_by_count,
    top_lender_share_by_amount,
    lender_hhi_by_amount,
    lender_hhi_by_count,
    lender_concentration_tier,
    first_approval_date,
    latest_approval_date
FROM INT.INT_SBA_LENDING_COUNTY_INDUSTRY_QTR
""")

sba.columns = sba.columns.str.lower()
show_basic_profile(sba)


## SBA activity over time

In [ ]:

period_activity = (
    sba.groupby(["year", "quarter", "period_id"])
    .agg(
        rows=("county_fips", "size"),
        states=("state_fips", "nunique"),
        counties=("county_fips", "nunique"),
        sectors=("qcew_industry_code", "nunique"),
        loans=("sba_loan_count", "sum"),
        gross_approval_amount=("sba_gross_approval_amount", "sum"),
        active_lenders=("active_lender_count", "sum"),
    )
    .reset_index()
    .sort_values(["year", "quarter"])
)

display(period_activity.tail(12))


In [ ]:

plt.figure(figsize=(9, 4.5))
plt.plot(period_activity["period_id"], period_activity["loans"], marker="o")
plt.title("SBA 7(a) loan count by period")
plt.xlabel("Period")
plt.ylabel("Loans")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(9, 4.5))
plt.plot(period_activity["period_id"], period_activity["gross_approval_amount"], marker="o")
plt.title("SBA 7(a) gross approval amount by period")
plt.xlabel("Period")
plt.ylabel("Gross approval amount")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Sector mix

In [ ]:

sector_activity = (
    sba.groupby(["qcew_industry_code", "sector_name"])
    .agg(
        rows=("county_fips", "size"),
        states=("state_fips", "nunique"),
        counties=("county_fips", "nunique"),
        loans=("sba_loan_count", "sum"),
        gross_approval_amount=("sba_gross_approval_amount", "sum"),
        active_lenders=("active_lender_count", "sum"),
        avg_hhi_amount=("lender_hhi_by_amount", "mean"),
        avg_top_lender_share_amount=("top_lender_share_by_amount", "mean"),
    )
    .reset_index()
    .sort_values("gross_approval_amount", ascending=False)
)

display(sector_activity)


In [ ]:

plt.figure(figsize=(9, 4.5))
plt.bar(sector_activity["sector_name"], sector_activity["gross_approval_amount"])
plt.title("SBA gross approval amount by sector")
plt.xlabel("Sector")
plt.ylabel("Gross approval amount")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Lender concentration

In [ ]:

concentration = (
    sba.groupby("lender_concentration_tier")
    .agg(
        rows=("county_fips", "size"),
        loans=("sba_loan_count", "sum"),
        gross_approval_amount=("sba_gross_approval_amount", "sum"),
        avg_hhi_amount=("lender_hhi_by_amount", "mean"),
        avg_top_lender_share_amount=("top_lender_share_by_amount", "mean"),
    )
    .reset_index()
    .sort_values("gross_approval_amount", ascending=False)
)

display(concentration)


In [ ]:

plt.figure(figsize=(8, 4.5))
plt.hist(sba["lender_hhi_by_amount"].dropna(), bins=30)
plt.title("Lender HHI by amount distribution")
plt.xlabel("Lender HHI by amount")
plt.ylabel("County-industry-quarter rows")
plt.tight_layout()
plt.show()


## Top lenders

In [ ]:

top_lenders = (
    sba.dropna(subset=["top_lender_name"])
    .groupby("top_lender_name")
    .agg(
        rows=("county_fips", "size"),
        loans=("top_lender_loan_count", "sum"),
        gross_approval_amount=("top_lender_gross_approval_amount", "sum"),
        states=("state_fips", "nunique"),
        counties=("county_fips", "nunique"),
        sectors=("qcew_industry_code", "nunique"),
    )
    .reset_index()
    .sort_values("gross_approval_amount", ascending=False)
    .head(25)
)

display(top_lenders)


## Geography / staging quality flags

This reads staging flags to inspect county/state and sector match quality at loan level.

In [ ]:

stg_quality = read_sql("""
SELECT
    has_state_fips,
    has_county_fips,
    county_state_fips_match,
    has_sector_mapping,
    COUNT(*) AS rows
FROM STG.STG_SBA_7A_LOANS
GROUP BY
    has_state_fips,
    has_county_fips,
    county_state_fips_match,
    has_sector_mapping
ORDER BY rows DESC
""")

stg_quality.columns = stg_quality.columns.str.lower()
display(stg_quality)


## Notes to capture

- Which sectors receive the most SBA support?
- Which county-industries have high growth but low SBA penetration after the mart join?
- Are lender concentration values interpretable and bounded?
- Are geography match rates acceptable for a portfolio project?